In [ ]:
from __future__ import annotations

from pathlib import Path
import os

import numpy as np
import pandas as pd
import pydicom
import torch
from skimage.transform import resize
from torch.utils.data import DataLoader, Dataset

### Data functions ###


def ct_normalization(img, meta):
    """
    Normaliza el rango HU a [0, 1].
    Ajustamos de -1024 (aire) a 3072 (tejido denso) para cubrir un rango amplio
    para resaltar detalles anatómicos.
    """
    def scale(img, meta, slope=1.0, intersept=-1024):

        slope = float(meta.get("Rescale Slope", slope))
        intersept = float(meta.get("Rescale Intercept", intersept))
        
        img = img * slope + intersept  # Aplica la transformación de escala y desplazamiento
        img = resize(img, (256, 256), preserve_range=True)  # Asegura que todas las imágenes tengan el mismo tamaño
        img = (img + 1024) / (3072 + 1024)  # Escala a [0, 1]
        return img.astype(np.float32)
    
    return scale(img, meta)


def _read_dicom(path: Path) -> np.ndarray:
    
    ds = pydicom.dcmread(path)  # Lee el archivo DICOM y convierte a numpy array
    metadata = {}
    for element in ds:
            if getattr(element, "name", None) == "Pixel Data":
                continue
            try:
                name = str(element.name)
                value = element.value
                if value is None:
                    continue
                metadata[name] = str(value)
            except Exception:
                continue

    return ct_normalization(ds.pixel_array, metadata)

def save_tensor_cache(tensor, cache_path: Path) -> None:
        if cache_path is None:
            return
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        tmp_path = cache_path.with_suffix(cache_path.suffix + ".tmp")
        torch.save(tensor, tmp_path)
        try:
            with open(tmp_path, "rb+") as handle:
                os.fsync(handle.fileno())
        except OSError:
            pass
        os.replace(tmp_path, cache_path)


class LDCTDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        dim: int = 2,
        model_type: str = "model",
    ) -> None:
        """
        Args:
            df: DataFrame with either:
                - dim == 2: columns ['Case', 'SDCT', 'LDCT'] where SDCT/LDCT are file paths to DICOM images
                - dim == 3: columns ['Case', 'SDCT', 'LDCT'] where SDCT/LDCT are volume directories
            transform: Optional transform to be applied on a sample.
            model_type: "model" or "vae", used to determine how to process the data.
        """
        self.dim = dim
        self.data = df
        self.model_type = model_type
        self.cache_saving = False

        self._preprocess()

    def _preprocess(self) -> pd.DataFrame:
        rows = []

        if self.model_type == "model":
            self.data.dropna(subset=['SDCT', 'LDCT'], inplace=True)
        elif self.model_type == "vae":
            self.data.dropna(subset=['SDCT'], inplace=True)

        unwrap_dir = lambda path: sorted([str(path / fname) for fname in os.listdir(path) if (path / fname).is_file()])

        idx = 0
        for row in self.data.itertuples(index=False):
            sdct_dir = Path(row.SDCT)
            sdct_files = unwrap_dir(sdct_dir)
            
            if self.model_type == "model":
                ldct_dir = Path(row.LDCT)
                ldct_files = unwrap_dir(ldct_dir)
                
                if self.dim == 3:
                    rows.append({
                        'Case': row.Case, 
                        'SDCT': sdct_files, 
                        'LDCT': ldct_files,
                        'split_count': len(ldct_files),
                        'split_idx': idx
                    })
                else:
                    rows.extend([
                        {'Case': row.Case, 'SDCT': s, 'LDCT': l, 'split_count': len(ldct_files), 'split_idx': idxS} 
                        for idxS, (s, l) in enumerate(zip(sdct_files, ldct_files))
                    ])
            
            elif self.model_type == "vae":
                if self.dim == 3:
                    rows.append({
                        'Case': row.Case, 
                        'SDCT': sdct_files,
                        'split_count': len(sdct_files),
                        'split_idx': idx
                    })
                else:
                    rows.extend([
                        {'Case': row.Case, 'SDCT': s, 'split_count': len(sdct_files), 'split_idx': idxS} 
                        for idxS, s in enumerate(sdct_files)
                    ])

            idx += 1
                    
        df = pd.DataFrame(rows)
        self.data = df.to_dict("records")
        self.size = len(self.data)
    
    def load_dicom(self, idx: int, modality: str = "SDCT") -> dict:
        row = self.data[idx]
        cache_path = "./cache/tensors"

        check_cache = lambda modality: True if Path(f"./cache/tensors/{self.data[idx]['Case']}_{modality}.pt").exists() else False

        if self.dim == 3:
            if check_cache(modality):
                cache_file = Path(f"./cache/tensors/{self.data[idx]['Case']}_{modality}.pt")
                return torch.load(cache_file)
            volume = np.array([_read_dicom(f) for f in sorted(row[modality])]) # Shape: (D, H, W)
            tensor = torch.as_tensor(volume).float().contiguous()  # Shape: (1, D, H, W)
            
            if self.cache_saving:
                save_tensor_cache(tensor, cache_path=Path(f"{cache_path}/{row['Case']}_{modality}").with_suffix('.pt'))
                
            return tensor
        else:
            if check_cache(modality):
                cache_file = Path(f"./cache/tensors/{self.data[idx]['Case']}_{modality}.pt")
                return torch.load(cache_file)
            img = _read_dicom(Path(row[modality]))
            tensor = torch.as_tensor(img).float().contiguous()  # Shape: (1, H, W)
            if self.cache_saving:
                save_tensor_cache(tensor, Path(f"{cache_path}/{row['split_idx']}_{row['Case']}_{modality}").with_suffix('.pt'))
            return tensor
    
    def __len__(self) -> int:
        return self.size

    def __getitem__(self, idx: int) -> dict:
        row = self.data[idx]
        sample = {'Case': row['Case']}
        sample['split_idx'] = row['split_idx']
        sample['size'] = (256, 256) if self.dim == 2 else (len(row['SDCT']), 256, 256)
        if self.model_type == "model":
            # sample['SDCT'] = row['SDCT']
            sample['SDCT'] = self.load_dicom(idx, modality='SDCT')
            # sample['LDCT'] = row['LDCT']
            sample['LDCT'] = self.load_dicom(idx, modality='LDCT')
        elif self.model_type == "vae":
            # sample['SDCT'] = row['SDCT']
            sample['SDCT'] = self.load_dicom(idx, modality='SDCT')
        
        return sample

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train = False
dataset = pd.read_csv(os.path.join("Dataset", 'train.txt' if train else 'test.txt'), sep='\t', names=('Case','SDCT','LDCT',))
print("Dataset loaded successfully.")
dataset = LDCTDataset(dataset, dim=2, model_type="vae")
print("Dataset object created successfully.")

In [ ]:
loader = DataLoader(dataset, batch_size=1, shuffle=True)
loader = iter(loader)  # Convertimos el DataLoader a un iterador para usar next()

In [ ]:
import torch.nn.functional as F

batch = next(loader)  # Obtenemos el primer batch
torch.cuda.empty_cache()
inputs = batch['SDCT'].to(device)  # Shape: (B, D, H, W)

divisor = 16
actual_d = inputs.shape[2]

# Calculamos cuánto falta para el siguiente múltiplo
pad_needed = (divisor - (actual_d % divisor)) % divisor

if pad_needed > 0:
    inputs = F.pad(inputs, (0, 0, 0, 0, 0, pad_needed), value=0)
    print(f"Se agregaron {pad_needed} cortes. Nueva profundidad: {inputs.shape[2]}")

In [ ]:
# Dimensiones deseadas del cubo
cube_size = 16

# Calculamos los puntos de inicio para centrar el recorte
# inputs shape: [B, C, D, H, W] -> [1, 1, 280, 256, 256]
d_start = (inputs.shape[2] - cube_size) // 2
h_start = (inputs.shape[3] - cube_size) // 2
w_start = (inputs.shape[4] - cube_size) // 2

# Extraemos el cubo usando slicing de PyTorch
perfect_cube = inputs[:, :, 
                      d_start : d_start + cube_size, 
                      h_start : h_start + cube_size, 
                      w_start : w_start + cube_size]

print(f"Forma del cubo: {perfect_cube.shape}") 
# Resultado: torch.Size([1, 1, 128, 128, 128])

In [ ]:
import torch
from medvae import MVAE
import matplotlib.pyplot as plt

# --- 1. Configuración del dispositivo ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# if torch.cuda.device_count() > 1:
#     print(f"Using {torch.cuda.device_count()} GPUs!")
#     model = nn.DataParallel(model)

# --- 2. Carga del Modelo ---
try:
    model = MVAE(model_name="medvae_4_1_3d", modality="ct").to(device)
    model.eval()
    print("¡Modelo MVAE cargado correctamente!")
except Exception as e:
    print(f"Error al cargar MVAE: {e}")

with torch.no_grad():
    # 1. Obtener el latente
    reconstruction, latent = model(inputs, decode=True)
    print(f"Dimensiones latentes:  {latent.shape} (Reducción 4x)")
    print(f"Dimensiones reconstrucción: {reconstruction.shape}")

In [ ]:
# Si agregaste 8 cortes al final, se los quitas así:
reconstruction = reconstruction[:actual_d, :, :]
# --- 5. Visualización ---
plt.figure(figsize=(15, 5))

# 1. Entrada (Suponiendo que inputs es [Slices, H, W] o [1, 1, Slices, H, W])
# Si inputs es [1, 1, 200, 256, 256], lo limpiamos con .squeeze()
input_clean = inputs.squeeze()

plt.subplot(1, 3, 1)
plt.title(f"Entrada (Slice {2})")
plt.imshow(input_clean[1].cpu().detach(), cmap="gray")
plt.axis("off")

# 2. Espacio Latente (Shape: [50, 64, 64])
# Tomamos la slice central del espacio comprimido
plt.subplot(1, 3, 2)
plt.title(f"Latente (Slice {2})")
plt.imshow(latent[1].cpu().detach(), cmap="magma")
plt.axis("off")

# 3. Reconstrucción (Shape: [200, 256, 256])
recon_clean = reconstruction.squeeze()

plt.subplot(1, 3, 3)
plt.title(f"Reconstrucción (Slice {2})")
plt.imshow(recon_clean[1].cpu().detach(), cmap="gray")
plt.axis("off")

plt.tight_layout()
plt.show()

# Prints de verificación
print(f"Forma final procesada:")
print(f"Entrada: {input_clean.shape}")
print(f"Latente: {latent.shape}")
print(f"Reconstrucción: {recon_clean.shape}")
torch.cuda.empty_cache()

In [ ]:
# 2. Espacio Latente (Shape: [50, 64, 64])
# Se plotea cada slice del espacio latente como un subplot en forma de matriz

plt.figure(figsize=(20, 20))

for i in range(latent.shape[0]//2):
    plt.subplot(10, 5, i + 1)  # Ajusta el número de filas y columnas según el número de slices
    plt.title(f"Latente Slice {i}")
    plt.imshow(latent[i].cpu().detach(), cmap="magma")
    plt.axis("off")

In [ ]:
# 2. Espacio Latente (Shape: [50, 64, 64])
# Se plotea cada slice del espacio latente como un subplot en forma de matriz

plt.figure(figsize=(20, 20))

for i in range(33):
    plt.subplot(10, 5, i + 1)  # Ajusta el número de filas y columnas según el número de slices
    plt.title(f"Reconstrucción Slice {i + 1}")
    plt.imshow(recon_clean[i].cpu().detach(), cmap="gray")
    plt.axis("off")